# Salary Prediction — Final Report

## Table of Contents

1. Introduction  
2. Dataset Overview  
3. Exploratory Data Analysis  
4. Feature Engineering  
5. Model Selection and Training  
6. Model Evaluation  
7. Model Assumptions and Interpretation  
8. From Model to Production: API Design (Conceptual)  
9. Conclusions


## 1. Introduction


Salary prediction is a common regression problem in people analytics and labor market studies. 
Understanding how demographic and professional attributes relate to compensation can support decision-making in areas such as workforce planning, benchmarking, and talent analytics.


## Objective
Build and evaluate regression models to predict **Salary** using demographic and professional features.

## Approach

- Load and merge the provided datasets (people, salary, descriptions).
- Establish a baseline using **DummyRegressor**.
- Train a **Linear Regression** model with a clean preprocessing pipeline.
- Run an additional experiment including **Job Title** and compare results.
- Evaluate using **MAE** (and bootstrap confidence intervals for robustness).

### Project Setup

This section configures the Python environment to allow importing project modules from the `src` directory.


In [1]:
import sys
from pathlib import Path

# Add project root to Python path
project_root = Path().resolve().parent
sys.path.append(str(project_root))


In [2]:
import pandas as pd

from src.data import load_and_merge_data
from src.features import build_preprocessor
from src.models import build_model
from src.evaluation import evaluate_holdout


## 2. Dataset Overview

The dataset used in this project is a public dataset containing demographic and professional information about individuals. It includes attributes related to personal background, education, and work experience, which are commonly used in salary prediction problems.

The target variable for this regression task is **Salary**, representing the annual compensation of an individual.

The dataset contains a mix of numerical and categorical features:

- **Numerical features**:  
  - Age  
  - Years of Experience  

- **Categorical features**:  
  - Gender  
  - Education Level  
  - Job Title  

This combination of feature types requires appropriate preprocessing steps, such as scaling for numerical variables and encoding for categorical variables, before training machine learning models.


In [3]:
PEOPLE_PATH_FILE = "data/raw/people.csv"
SALARY_PATH_FILE = "data/raw/salary.csv"
DESCRIPTIONS_PATH_FILE = "data/raw/descriptions.csv"


In [4]:
df = load_and_merge_data(
    people_path=PEOPLE_PATH_FILE,
    salary_path=SALARY_PATH_FILE,
    descriptions_path=DESCRIPTIONS_PATH_FILE
)

print("Dataset shape:", df.shape)
df.head()


Dataset shape: (373, 8)


,id,Age,Gender,Education Level,Job Title,Years of Experience,Salary,Description
0,0,32.0,Male,Bachelor's,Software Engineer,5.0,90000.0,I am a 32-year-old male working as a Software ...
1,1,28.0,Female,Master's,Data Analyst,3.0,65000.0,I am a 28-year-old data analyst with a Master'...
2,2,45.0,Male,PhD,Senior Manager,15.0,150000.0,I am a 45-year-old Senior Manager with a PhD a...
3,3,36.0,Female,Bachelor's,Sales Associate,7.0,60000.0,I am a 36-year-old female Sales Associate with...
4,4,52.0,Male,Master's,Director,20.0,200000.0,I am a 52-year-old male with over two decades ...


## 3. Exploratory Data Analysis

A comprehensive exploratory data analysis (EDA) was performed in a separate notebook (`01_eda.ipynb`). 
This final report focuses on summarizing the most relevant insights that directly informed feature selection and model design.


Key insights from the EDA include:

- Salary shows a right-skewed distribution with a small number of high-income outliers.
- Years of Experience exhibits a strong positive and approximately linear relationship with Salary.
- Categorical variables such as Education Level and Job Title introduce meaningful differences in salary distributions.


## 4. Feature Engineering

Feature engineering and preprocessing were implemented to transform raw input variables into a format suitable for machine learning models. The objective was to ensure consistency, reproducibility, and correct handling of mixed data types.

Numerical features (**Age**, **Years of Experience**) were scaled to standardize their ranges and prevent features with larger magnitudes from disproportionately influencing the model.

Categorical features (**Gender**, **Education Level**, **Job Title**) were encoded to convert non-numerical values into numerical representations that can be consumed by regression models.

The dataset contains a small number of missing values in some categorical features, which were handled during preprocessing.

All preprocessing steps were encapsulated within a single scikit-learn **Pipeline**, ensuring that the same transformations applied during training are also applied during inference. This design choice reduces the risk of data leakage and improves maintainability.


## 5. Model Selection and Training

Model selection followed an incremental approach, starting with a simple baseline to establish a reference level of performance.

A **DummyRegressor** was used as the baseline model. This model predicts the mean value of the target variable regardless of the input features, providing a lower bound against which more informative models can be compared.

The main predictive model selected for this project was **Linear Regression**, trained using the preprocessing pipeline described previously. This model was chosen due to its interpretability, robustness, and its suitability for problems where an approximately linear relationship exists between features and the target variable.

In addition, an alternative configuration was evaluated by including **Job Title** as a categorical feature, with the goal of assessing whether more granular professional information could improve predictive performance.


## 6. Model Evaluation

Model performance was evaluated using **Mean Absolute Error (MAE)** as the primary metric. MAE was chosen because it provides an intuitive measure of average prediction error in the same units as the target variable (salary).

To assess the robustness of the results, **bootstrap confidence intervals** were computed on the test set, allowing performance to be reported as a range rather than a single point estimate.


### 6.1 Baseline Model (DummyRegressor)

A DummyRegressor was used as a naive baseline by predicting the mean salary for all observations. This establishes a lower bound for performance.


In [ ]:
pre_base = build_preprocessor(include_job_title=False)
pipe_dummy = build_model("dummy", pre_base)

res_dummy = evaluate_holdout(df, target_col="Salary", pipeline=pipe_dummy, with_ci=True)
res_dummy


NameError: name 'build_preprocessor' is not defined

### 6.2 Linear Regression (without Job Title)

A linear regression model was trained using the preprocessing pipeline. This model serves as an interpretable baseline that leverages demographic and education-related features.


In [7]:
pipe_linear_base = build_model("linear", pre_base)

res_linear_base = evaluate_holdout(df, target_col="Salary", pipeline=pipe_linear_base, with_ci=True)
res_linear_base


{'mae': 10797.999241268952,
 'n_train': 298,
 'n_test': 75,
 'mae_boot_mean': 10798.169699093474,
 'mae_ci_low': 8539.53484842004,
 'mae_ci_high': 13626.336524585484}

### 6.3 Linear Regression (with Job Title)

An additional experiment included Job Title as a categorical feature to test whether more granular professional information improves predictive performance.


In [8]:
pre_job = build_preprocessor(include_job_title=True)
pipe_linear_job = build_model("linear", pre_job)

res_linear_job = evaluate_holdout(df, target_col="Salary", pipeline=pipe_linear_job, with_ci=True)
res_linear_job


{'mae': 11371.572373301655,
 'n_train': 298,
 'n_test': 75,
 'mae_boot_mean': 11354.087479701811,
 'mae_ci_low': 8769.990362862456,
 'mae_ci_high': 14839.644655977329}

### 6.4 Summary and Comparison

The table below summarizes MAE performance and bootstrap confidence intervals for each model configuration. Lower MAE indicates better predictive accuracy.


In [9]:
results = pd.DataFrame([
    {"model": "Dummy (baseline)", **res_dummy},
    {"model": "Linear (no Job Title)", **res_linear_base},
    {"model": "Linear (+ Job Title)", **res_linear_job},
])

cols_order = ["model", "mae", "mae_ci_low", "mae_ci_high", "n_train", "n_test"]
results[cols_order].sort_values("mae")


,model,mae,mae_ci_low,mae_ci_high,n_train,n_test
1,Linear (no Job Title),10797.999241,8539.534848,13626.336525,298,75
2,Linear (+ Job Title),11371.572373,8769.990363,14839.644656,298,75
0,Dummy (baseline),40533.333333,34463.333333,47133.333333,298,75


**Interpretation**

- The **DummyRegressor** baseline yields an MAE of ~40k, which indicates that naive mean prediction performs poorly for this task.
- The **Linear Regression (no Job Title)** model reduces MAE to ~10.8k, showing that demographic and education-related features provide strong predictive signal.
- Adding **Job Title** does not improve results in this split (MAE ~11.4k). Given the overlapping confidence intervals and the increased feature complexity, the simpler linear model without Job Title is selected as the final model for this report.


## 7. Model Assumptions and Interpretation

The final model is based on Linear Regression, which relies on several underlying assumptions. While no formal statistical tests were performed, these assumptions were considered qualitatively during analysis.

A key assumption is the existence of an approximately linear relationship between the input features and the target variable. Exploratory analysis conducted in a separate notebook indicates that **Years of Experience** shows a strong and near-linear relationship with **Salary**, supporting the suitability of a linear model.

Another assumption is the independence of observations. This is considered reasonable, as each row in the dataset represents an individual employee record.

Finally, some degree of heteroscedasticity is expected in salary prediction problems, as salary variability tends to increase with experience and seniority. This does not invalidate the model but suggests that prediction uncertainty may vary across different salary ranges.

Overall, the model assumptions are sufficiently met for the purpose of this project, and predictions should be interpreted as approximate estimates rather than exact values.


## 8. From Model to Production: API Design (Conceptual)

Although this project focuses on model development and evaluation, a natural next step in a real-world scenario would be deploying the trained model so it can be consumed by other systems.

The trained preprocessing and modeling pipeline can be serialized using tools such as `joblib` or `pickle`, ensuring that the same transformations applied during training are also applied during inference.

A simple REST API could be implemented using **FastAPI**, exposing an endpoint that receives employee attributes in JSON format and returns a salary prediction.

Example request:
```json
{
  "id": 368,
  "age": 35,
  "gender": "Male",
  "education_level": "Master",
  "job_title": "Data Analyst",
  "years_of_experience": 7
}

Example response:

{
  "predicted_salary": 108000
}

## 9. Conclusions

This project developed a complete and structured workflow to address a salary prediction problem using demographic and professional data.

Starting from a naive baseline model, progressively more informative models were evaluated using a consistent preprocessing and evaluation framework. The Linear Regression model substantially improved performance over the baseline, confirming that core demographic and education-related features capture meaningful information about salary levels.

An additional experiment incorporating Job Title was explored to assess the trade-off between model complexity and predictive performance. While this feature adds richer professional context, the results did not justify its inclusion given the similar performance and overlapping confidence intervals.

Overall, this work demonstrates a reproducible and well-organized approach to model development, emphasizing interpretability, robustness, and clear decision-making over unnecessary complexity. The resulting model provides reliable approximate salary estimates and can serve as a strong foundation for further improvements or production deployment.
